In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import sklearn
import warnings
warnings.filterwarnings("ignore")

from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, misc_utils, responsecomparison_utils as rc_utils
from compcats.localizer import  localizer

import main_analysis as ma
import paths

splithalf_csv = pd.read_csv(f'{ma.SUBJECTSUBJECT_CORRELATIONS_CSV_PATH}')

# Storing results
RIDGE_LESION_EM_CSV_PATH = f'{ma.RESULTS_DIR}/nsd515_lesion_ems_ridge.csv'
L1POS_LESION_EM_CSV_PATH = f'{ma.RESULTS_DIR}/nsd515_lesion_ems_l1pos.csv'
csv_path = RIDGE_LESION_EM_CSV_PATH
if os.path.exists(csv_path):
    results_csv = pd.read_csv(csv_path)
    results_csv_list = results_csv.to_dict('records')
else:
    results_csv_list = []


alpha_values = np.array([1e-3, 1e-2, 1e-1, 1e-0, 1e+1, 1e+2, 1e+3])
n_splits = 5
seed = 42
trainval_kf = sklearn.model_selection.KFold(n_splits=n_splits, shuffle=True, random_state=seed)

tthreshold = 7
model_localizer = 'grillspector'

train_subset = 485
test_subset = 515
frois = ['FFA', 'PPA', 'EBA']

subjects = [1, 2, 5, 7]

for froi in frois:
    for model_name in ma.get_model_names():
        model = ma.get_model(model_name)
        layer = ma.get_froi_models_layers(froi, froi)[model_name]
        for subject in subjects:
            row_params = {
                'model': model_name,
                'subject': subject,
                'layer': layer,
                'model_froi': froi,
                'fmri_froi': froi,
                'model_tthreshold': tthreshold,
                'fmri_tthreshold': tthreshold,
                'model_localizer': model_localizer,
                'mapper': 'ridge'
            }
            # Check if row is present
            if misc_utils.check_row_present(row_params, results_csv_list):
                continue
            results_dict = row_params.copy()

            Y_trainval, _ = nsd.get_betas(
                paths.NSD_DIR, subject, froi, tthreshold, subset=train_subset, voxels=True
            )
            Y_trainval = Y_trainval.cpu().numpy()
            Y_test, _ = nsd.get_betas(
                paths.NSD_DIR, subject, froi, tthreshold, subset=test_subset, voxels=True
            )

            # Splithalf correlations
            splithalf_info = splithalf_csv[
                (splithalf_csv['fmri_froi'] == froi) &
                (splithalf_csv['fmri_tthreshold'] == tthreshold) &
                (splithalf_csv['fmri_dataset'] == f'nsd{test_subset}') &
                (splithalf_csv['subject1'] == subject)
            ].iloc[0]
            splithalf_pearsonr = splithalf_info['subject1_splithalf_pearsonr'].item()
            splithalf_rdmr = splithalf_info['subject1_splithalf_rdmr'].item()
            results_dict['fmri_splithalf_pearsonr'] = splithalf_pearsonr
            results_dict['fmri_splithalf_rdmr'] = splithalf_rdmr

            X_trianval_layer = model_utils.get_dataset_activations(
                ma.MODELS_DIR, paths.NSD_DIR, model, ma.get_froi_models_layers(froi, froi)[model_name], 
                'nsd', subset=train_subset
            )
            X_test_layer = model_utils.get_dataset_activations(
                ma.MODELS_DIR, paths.NSD_DIR, model, ma.get_froi_models_layers(froi, froi)[model_name], 
                'nsd', subset=test_subset
            )
            model_mask_froi = localizer.get_froi_mask(
                ma.get_localizer_dir(model_localizer), model_localizer, ma.MODELS_DIR, model, layer, froi, tthreshold
            )
            model_mask_froi_t0 = localizer.get_froi_mask(
                ma.get_localizer_dir(model_localizer), model_localizer, ma.MODELS_DIR, model, layer, froi, 0
            )
            model_mask_froi_lesioned = ~model_mask_froi_t0

            print(f'FROI: {froi}, Model: {model_name}, Subject: {subject}, mfROI -> fROI')
            X_trainval = X_trianval_layer[:, model_mask_froi]
            X_test = X_test_layer[:, model_mask_froi]
            X_trainval = X_trainval.cpu().numpy()
            X_test = X_test.cpu().numpy()
            mapping = sklearn.linear_model.RidgeCV(
                alphas=alpha_values,
                cv=trainval_kf,
                scoring='r2',
            )
            map = mapping.fit(X_trainval, Y_trainval)
            y_pred = torch.FloatTensor(mapping.predict(X_test))
            # Get test performances
            pearsonr = rc_utils.pearsonr(Y_test.mean(axis=-1), y_pred.mean(axis=-1))
            rdmr = rc_utils.rdmr(Y_test, y_pred)
            # Store results
            results_dict['froi_raw_pearsonr'] = pearsonr
            results_dict['froi_normed_pearsonr'] = (pearsonr / np.sqrt(splithalf_pearsonr)).item()
            results_dict['froi_raw_rdmr'] = rdmr
            results_dict['froi_normed_rdmr'] = (rdmr / np.sqrt(splithalf_rdmr)).item()
            
            print(f'FROI: {froi}, Model: {model_name}, Subject: {subject}, mfROI lesioned -> fROI')
            X_trainval = X_trianval_layer[:, model_mask_froi_lesioned]
            X_test = X_test_layer[:, model_mask_froi_lesioned]
            X_trainval = X_trainval.cpu().numpy()
            X_test = X_test.cpu().numpy()
            mapping = sklearn.linear_model.RidgeCV(
                alphas=alpha_values,
                cv=trainval_kf,
                scoring='r2',
            )
            map = mapping.fit(X_trainval, Y_trainval)
            y_pred = torch.FloatTensor(mapping.predict(X_test))
            # Get test performances
            pearsonr = rc_utils.pearsonr(Y_test.mean(axis=-1), y_pred.mean(axis=-1))
            rdmr = rc_utils.rdmr(Y_test, y_pred)
            # Store results
            results_dict['lesion_raw_pearsonr'] = pearsonr
            results_dict['lesion_normed_pearsonr'] = (pearsonr / np.sqrt(splithalf_pearsonr)).item()
            results_dict['lesion_raw_rdmr'] = rdmr
            results_dict['lesion_normed_rdmr'] = (rdmr / np.sqrt(splithalf_rdmr)).item()

            # Save results
            results_csv_list.append(results_dict)
            results_csv = pd.DataFrame(results_csv_list)
            results_csv.to_csv(
                csv_path, index=False
            )

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import sklearn
import warnings
warnings.filterwarnings("ignore")

from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, misc_utils, responsecomparison_utils as rc_utils
from compcats.localizer import  localizer

import main_analysis as ma
import paths

splithalf_csv = pd.read_csv(f'{ma.SUBJECTSUBJECT_CORRELATIONS_CSV_PATH}')

# Storing results
RIDGE_LESION_EM_CSV_PATH = f'{ma.RESULTS_DIR}/nsd515_lesion_ems_ridge.csv'
L1POS_LESION_EM_CSV_PATH = f'{ma.RESULTS_DIR}/nsd515_lesion_ems_l1pos.csv'
csv_path = L1POS_LESION_EM_CSV_PATH
if os.path.exists(csv_path):
    results_csv = pd.read_csv(csv_path)
    results_csv_list = results_csv.to_dict('records')
else:
    results_csv_list = []


alpha_values = np.array([1e-3, 1e-2, 1e-1, 1e-0, 1e+1, 1e+2, 1e+3])
n_splits = 5
seed = 42
trainval_kf = sklearn.model_selection.KFold(n_splits=n_splits, shuffle=True, random_state=seed)

tthreshold = 7
model_localizer = 'grillspector'

train_subset = 485
test_subset = 515
frois = ['FFA', 'PPA', 'EBA']

subjects = [1, 2, 5, 7]

for froi in frois:
    for model_name in ma.get_model_names():
        model = ma.get_model(model_name)
        layer = ma.get_froi_models_layers(froi, froi)[model_name]
        for subject in subjects:
            row_params = {
                'model': model_name,
                'subject': subject,
                'layer': layer,
                'model_froi': froi,
                'fmri_froi': froi,
                'model_tthreshold': tthreshold,
                'fmri_tthreshold': tthreshold,
                'model_localizer': model_localizer,
                'mapper': 'l1pos'
            }
            # Check if row is present
            if misc_utils.check_row_present(row_params, results_csv_list):
                continue
            results_dict = row_params.copy()

            Y_trainval, _ = nsd.get_betas(
                paths.NSD_DIR, subject, froi, tthreshold, subset=train_subset, voxels=True
            )
            Y_trainval = Y_trainval.cpu().numpy()
            Y_test, _ = nsd.get_betas(
                paths.NSD_DIR, subject, froi, tthreshold, subset=test_subset, voxels=True
            )

            # Splithalf correlations
            splithalf_info = splithalf_csv[
                (splithalf_csv['fmri_froi'] == froi) &
                (splithalf_csv['fmri_tthreshold'] == tthreshold) &
                (splithalf_csv['fmri_dataset'] == f'nsd{test_subset}') &
                (splithalf_csv['subject1'] == subject)
            ].iloc[0]
            splithalf_pearsonr = splithalf_info['subject1_splithalf_pearsonr'].item()
            splithalf_rdmr = splithalf_info['subject1_splithalf_rdmr'].item()
            results_dict['fmri_splithalf_pearsonr'] = splithalf_pearsonr
            results_dict['fmri_splithalf_rdmr'] = splithalf_rdmr

            X_trianval_layer = model_utils.get_dataset_activations(
                ma.MODELS_DIR, paths.NSD_DIR, model, ma.get_froi_models_layers(froi, froi)[model_name], 
                'nsd', subset=train_subset
            )
            X_test_layer = model_utils.get_dataset_activations(
                ma.MODELS_DIR, paths.NSD_DIR, model, ma.get_froi_models_layers(froi, froi)[model_name], 
                'nsd', subset=test_subset
            )
            model_mask_froi = localizer.get_froi_mask(
                ma.get_localizer_dir(model_localizer), model_localizer, ma.MODELS_DIR, model, layer, froi, tthreshold
            )
            model_mask_froi_t0 = localizer.get_froi_mask(
                ma.get_localizer_dir(model_localizer), model_localizer, ma.MODELS_DIR, model, layer, froi, 0
            )
            model_mask_froi_lesioned = ~model_mask_froi_t0

            print(f'FROI: {froi}, Model: {model_name}, Subject: {subject}, mfROI -> fROI')
            X_trainval = X_trianval_layer[:, model_mask_froi]
            X_test = X_test_layer[:, model_mask_froi]
            X_trainval = X_trainval.cpu().numpy()
            X_test = X_test.cpu().numpy()
            mapping = sklearn.linear_model.LassoCV(
                alphas=alpha_values,
                cv=trainval_kf,
                scoring='r2',
                n_jobs=-1,
                positive=True,
                random_state=seed,
            )
            map = mapping.fit(X_trainval, Y_trainval)
            y_pred = torch.FloatTensor(mapping.predict(X_test))
            # Get test performances
            pearsonr = rc_utils.pearsonr(Y_test.mean(axis=-1), y_pred.mean(axis=-1))
            rdmr = rc_utils.rdmr(Y_test, y_pred)
            # Store results
            results_dict['froi_raw_pearsonr'] = pearsonr
            results_dict['froi_normed_pearsonr'] = (pearsonr / np.sqrt(splithalf_pearsonr)).item()
            results_dict['froi_raw_rdmr'] = rdmr
            results_dict['froi_normed_rdmr'] = (rdmr / np.sqrt(splithalf_rdmr)).item()
            
            print(f'FROI: {froi}, Model: {model_name}, Subject: {subject}, mfROI lesioned -> fROI')
            X_trainval = X_trianval_layer[:, model_mask_froi_lesioned]
            X_test = X_test_layer[:, model_mask_froi_lesioned]
            X_trainval = X_trainval.cpu().numpy()
            X_test = X_test.cpu().numpy()
            mapping = sklearn.linear_model.LassoCV(
                alphas=alpha_values,
                cv=trainval_kf,
                scoring='r2',
                n_jobs=-1,
                positive=True,
                random_state=seed,
            )
            map = mapping.fit(X_trainval, Y_trainval)
            y_pred = torch.FloatTensor(mapping.predict(X_test))
            # Get test performances
            pearsonr = rc_utils.pearsonr(Y_test.mean(axis=-1), y_pred.mean(axis=-1))
            rdmr = rc_utils.rdmr(Y_test, y_pred)
            # Store results
            results_dict['lesion_raw_pearsonr'] = pearsonr
            results_dict['lesion_normed_pearsonr'] = (pearsonr / np.sqrt(splithalf_pearsonr)).item()
            results_dict['lesion_raw_rdmr'] = rdmr
            results_dict['lesion_normed_rdmr'] = (rdmr / np.sqrt(splithalf_rdmr)).item()

            # Save results
            results_csv_list.append(results_dict)
            results_csv = pd.DataFrame(results_csv_list)
            results_csv.to_csv(
                csv_path, index=False
            )